# 📧 Gmail RAG — Multi-Account Edition
Supports **multiple Gmail accounts** for a single user with:
- ✅ Local embeddings via `sentence-transformers` (no API rate limits)
- ✅ Separate OAuth token per account
- ✅ Sliding-window chunking (handles long emails)
- ✅ Retry logic with backoff for Gmail API calls
- ✅ Pagination support (fetch beyond 100 emails)
- ✅ Gemini 2.5 Flash for final answer generation only

In [ ]:
# ── Cell 1: Install all dependencies ──────────────────────────────────────────
!pip install -q groq numpy google-auth-oauthlib google-auth-httplib2 google-api-python-client sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.5 MB/s eta 0:00:00


In [ ]:
# ── Cell 2: Upload your OAuth credentials file ─────────────────────────────────
from google.colab import files
uploaded = files.upload()
# A file picker will appear — select your client_secret_xxxx.json file

Saving client_secret_817999543115-r08qv85q4ebqvgo2t9tsmj8ur6mpj39l.apps.googleusercontent.com.json to client_secret_817999543115-r08qv85q4ebqvgo2t9tsmj8ur6mpj39l.apps.googleusercontent.com.json


In [ ]:
# ── Cell 3: Rename credentials file ───────────────────────────────────────────
import os

# Auto-detect the uploaded credentials file
cred_file = [f for f in os.listdir('.') if f.startswith('client_secret') and f.endswith('.json')]
if cred_file:
    os.rename(cred_file[0], 'credentials.json')
    print(f"Renamed '{cred_file[0]}' → credentials.json ✅")
else:
    print("credentials.json already exists ✅" if os.path.exists('credentials.json') else "❌ No credentials file found — re-upload in Cell 2")

Renamed 'client_secret_817999543115-r08qv85q4ebqvgo2t9tsmj8ur6mpj39l.apps.googleusercontent.com.json' → credentials.json ✅


In [ ]:
# ── Cell 4: Setup Gemini client (used only for final answer generation) ────────
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get('GROQ_API')  # Add this secret in Colab
client = Groq(api_key=GROQ_API_KEY)

# Quick test
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Reply with: Groq is ready!"}],
    max_tokens=50
)
print(response.choices[0].message.content)

Groq is ready!


In [ ]:
# ── Cell 5: Load local embedding model (no API limits, runs on Colab GPU/CPU) ──
from sentence_transformers import SentenceTransformer

print("Loading embedding model (one-time download ~90MB)...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready ✅")

Loading embedding model (one-time download ~90MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready ✅


In [ ]:
# ── Cell 6: Multi-account Gmail OAuth ─────────────────────────────────────────
# ⚠️  ADD YOUR GMAIL ACCOUNTS BELOW — all accounts must be added as
#     test users in your Google Cloud OAuth consent screen.
# ─────────────────────────────────────────────────────────────────
GMAIL_ACCOUNTS = [
    "apekshakor05@gmail.com",
    "apekshakor07@gmail.com",
    # "add_more@gmail.com",
]
# ─────────────────────────────────────────────────────────────────

import os, pickle
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.auth.transport.requests import Request

SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

def get_gmail_service(account_email):
    """Authenticate one Gmail account and return its service object."""
    # Each account gets its own token file so they don't overwrite each other
    safe_name = account_email.replace('@', '_at_').replace('.', '_')
    token_file = f"token_{safe_name}.pickle"

    creds = None
    if os.path.exists(token_file):
        with open(token_file, 'rb') as f:
            creds = pickle.load(f)

    # Refresh if expired
    if creds and creds.expired and creds.refresh_token:
        try:
            creds.refresh(Request())
            with open(token_file, 'wb') as f:
                pickle.dump(creds, f)
            print(f"  Token refreshed for {account_email} ✅")
            return build('gmail', 'v1', credentials=creds)
        except Exception:
            creds = None  # Force re-auth if refresh fails

    if not creds or not creds.valid:
        flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
        flow.redirect_uri = 'urn:ietf:wg:oauth:2.0:oob'

        # login_hint pre-fills the Google account picker for this specific email
        auth_url, _ = flow.authorization_url(
            prompt='consent',
            access_type='offline',
            login_hint=account_email
        )

        print(f"\n{'='*65}")
        print(f"  Authorizing: {account_email}")
        print(f"  Open this URL in your browser (sign in as {account_email}):")
        print(f"\n  {auth_url}\n")
        print('='*65)

        code = input(f"  Paste the authorization code for {account_email}: ").strip()
        flow.fetch_token(code=code)
        creds = flow.credentials

        with open(token_file, 'wb') as f:
            pickle.dump(creds, f)

    return build('gmail', 'v1', credentials=creds)


# Authenticate all accounts
services = {}
for account in GMAIL_ACCOUNTS:
    print(f"\nConnecting: {account}")
    try:
        services[account] = get_gmail_service(account)
        print(f"✅ Connected: {account}")
    except Exception as e:
        print(f"❌ Failed to connect {account}: {e}")

print(f"\n{len(services)}/{len(GMAIL_ACCOUNTS)} accounts connected ✅")


Connecting: apekshakor05@gmail.com

  Authorizing: apekshakor05@gmail.com
  Open this URL in your browser (sign in as apekshakor05@gmail.com):

  https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=817999543115-r08qv85q4ebqvgo2t9tsmj8ur6mpj39l.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=6VdujsZwDUKA1Sj7pCc7sIxpcdt4XI&code_challenge=gV0NVPf9lk9vRoVBehG-Is0AdUuBGj9jsVYmZcyIMcI&code_challenge_method=S256&prompt=consent&access_type=offline&login_hint=apekshakor05%40gmail.com

  Paste the authorization code for apekshakor05@gmail.com: 4/1Aci98E_cKSeo6BNKPnL5Rg3IlXIHsg542HLswCtZMLqmekD4GX2kUgn-Rog
✅ Connected: apekshakor05@gmail.com

Connecting: apekshakor07@gmail.com

  Authorizing: apekshakor07@gmail.com
  Open this URL in your browser (sign in as apekshakor07@gmail.com):

  https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=817999543115-r08qv85q4ebqv

In [ ]:
# ── Cell 7: Fetch emails from ALL accounts ─────────────────────────────────────
import base64, time

# ── CONFIG ─────────────────────────────────────────────
MAX_EMAILS_PER_ACCOUNT = 100   # increase up to 500
GMAIL_QUERY = ''               # e.g. 'is:unread', 'from:boss@company.com', ''
# ───────────────────────────────────────────────────────

def extract_body(payload, max_chars=2000):
    """Recursively extract plain-text body from a Gmail message payload."""
    if 'parts' in payload:
        for part in payload['parts']:
            # Try this part
            if part.get('mimeType') == 'text/plain' and 'data' in part.get('body', {}):
                return base64.urlsafe_b64decode(
                    part['body']['data']
                ).decode('utf-8', errors='ignore')[:max_chars]
            # Recurse into nested multipart
            result = extract_body(part, max_chars)
            if result:
                return result
    elif 'data' in payload.get('body', {}):
        return base64.urlsafe_b64decode(
            payload['body']['data']
        ).decode('utf-8', errors='ignore')[:max_chars]
    return ''


def fetch_emails_from_account(service, account_email, query='', max_results=100):
    """Fetch emails with pagination support and retry logic."""
    emails = []
    page_token = None
    remaining = max_results

    while remaining > 0:
        fetch_count = min(remaining, 100)  # Gmail API max per page is 100

        # Retry up to 3 times on transient errors
        for attempt in range(3):
            try:
                list_kwargs = {
                    'userId': 'me',
                    'q': query,
                    'maxResults': fetch_count
                }
                if page_token:
                    list_kwargs['pageToken'] = page_token

                results = service.users().messages().list(**list_kwargs).execute()
                break
            except Exception as e:
                wait = 2 ** attempt * 3
                print(f"  List error (attempt {attempt+1}/3), retrying in {wait}s: {e}")
                time.sleep(wait)
        else:
            print(f"  Giving up on page for {account_email}")
            break

        messages = results.get('messages', [])
        if not messages:
            break

        for msg in messages:
            for attempt in range(3):
                try:
                    detail = service.users().messages().get(
                        userId='me', id=msg['id'], format='full'
                    ).execute()
                    break
                except Exception as e:
                    wait = 2 ** attempt * 2
                    print(f"  Fetch error, retrying in {wait}s: {e}")
                    time.sleep(wait)
            else:
                continue  # skip this message if all retries failed

            headers = {
                h['name']: h['value']
                for h in detail['payload'].get('headers', [])
            }
            body = extract_body(detail['payload'])

            emails.append({
                'account': account_email,
                'from':    headers.get('From', 'Unknown'),
                'to':      headers.get('To', ''),
                'subject': headers.get('Subject', 'No Subject'),
                'date':    headers.get('Date', ''),
                'body':    body.strip()
            })

        remaining -= len(messages)
        page_token = results.get('nextPageToken')
        if not page_token:
            break  # No more pages

        time.sleep(0.3)  # Gentle rate limiting between pages

    print(f"  [{account_email}] Fetched {len(emails)} emails ✅")
    return emails


# Fetch from all connected accounts
all_emails = []
for account, service in services.items():
    account_emails = fetch_emails_from_account(
        service, account,
        query=GMAIL_QUERY,
        max_results=MAX_EMAILS_PER_ACCOUNT
    )
    all_emails.extend(account_emails)

print(f"\n📬 Total emails across all accounts: {len(all_emails)}")

  [apekshakor05@gmail.com] Fetched 100 emails ✅
  [apekshakor07@gmail.com] Fetched 100 emails ✅

📬 Total emails across all accounts: 200


In [ ]:
# ── Cell 7b: Extract calendar events — ICS + keyword filter + batch Gemini + retry ──
import base64, json, re, time, random
from datetime import datetime

# ── Step 1: ICS parser (zero Gemini tokens) ─────────────────────────────────
try:
    from icalendar import Calendar as iCal
    ICAL_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'icalendar'], check=True)
    from icalendar import Calendar as iCal
    ICAL_AVAILABLE = True

def parse_ics_from_payload(payload):
    """Walk email parts looking for text/calendar or .ics attachments."""
    parts = payload.get('parts', [payload])
    for part in parts:
        mime     = part.get('mimeType', '')
        filename = part.get('filename', '')
        if mime == 'text/calendar' or filename.endswith('.ics'):
            data_b64 = part.get('body', {}).get('data', '')
            if not data_b64:
                continue  # large attachment via attachmentId — skip
            raw = base64.urlsafe_b64decode(data_b64)
            try:
                cal = iCal.from_ical(raw)
                for component in cal.walk():
                    if component.name == "VEVENT":
                        dtstart = component.get('dtstart')
                        start   = dtstart.dt if dtstart else None
                        return {
                            'has_event':   True,
                            'title':       str(component.get('summary', 'Untitled')),
                            'event_date':  str(start.date()) if hasattr(start, 'date') else str(start),
                            'event_time':  start.strftime('%H:%M') if hasattr(start, 'hour') else None,
                            'location':    str(component.get('location', '')),
                            'attendees':   [
                                str(a) for a in (
                                    component.get('attendee', [])
                                    if isinstance(component.get('attendee'), list)
                                    else [component.get('attendee', '')]
                                )
                            ],
                            'description': str(component.get('description', ''))[:200],
                            'source':      'ics',
                        }
            except Exception:
                pass
        # recurse into nested multipart
        if 'parts' in part:
            result = parse_ics_from_payload(part)
            if result:
                return result
    return None


# ── Step 2: Keyword pre-filter (skip obvious non-events) ────────────────────
EVENT_KEYWORDS = [
    'meeting', 'invite', 'invitation', 'calendar', 'schedule', 'scheduled',
    'call', 'zoom', 'teams', 'webex', 'google meet', 'appointment',
    '.ics', 'reminder', 'rsvp', 'join us', 'standup', 'sync', 'interview',
    'conference', 'webinar', 'event', 'session', 'workshop', 'demo',
]

def likely_has_event(email):
    text = (email.get('subject', '') + ' ' + email.get('body', '')).lower()
    return any(kw in text for kw in EVENT_KEYWORDS)


# ── Step 3: Batch Gemini extractor with retry + model cascade ───────────────
BATCH_SIZE  = 8          # slightly smaller = less timeout risk
MODEL_ORDER = [
    "llama-3.1-8b-instant",      # fast & cheap for structured extraction
    "llama-3.3-70b-versatile",   # fallback if 8B fails
]
MAX_RETRIES = 4
BASE_WAIT   = 2.0        # seconds — doubles each retry (exponential backoff)

def extract_events_batch(emails_batch):
    """
    Send up to BATCH_SIZE emails in one Gemini call.
    Retries with exponential backoff on 503.
    Falls back to next model on 429 (quota exhausted).
    Returns list of event dicts.
    """
    combined = ""
    for i, em in enumerate(emails_batch):
        combined += (
            f"\n\n--- EMAIL_INDEX {i} ---\n"
            f"From: {em['from']}\nDate: {em['date']}\n"
            f"Subject: {em['subject']}\nBody: {em['body'][:600]}"
        )

    prompt = f"""You are a calendar event extractor. Read the emails below and extract meetings, events, or scheduled calls.

{combined}

Respond ONLY with a valid JSON array — one object per email that has an event. Skip emails with no event.
Each object must have exactly these keys:
{{
  "email_index": <integer matching EMAIL_INDEX above>,
  "has_event": true,
  "title": "event title",
  "event_date": "YYYY-MM-DD or null",
  "event_time": "HH:MM or null",
  "location": "location or null",
  "attendees": ["email1"] or [],
  "description": "one line summary or null"
}}

If NO emails have events, return an empty array: []"""

    last_error = None

    for model in MODEL_ORDER:
        wait = BASE_WAIT
        for attempt in range(MAX_RETRIES):
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=1000,
                    temperature=0.0   # deterministic for JSON extraction
                )
                raw = response.choices[0].message.content.strip()

                raw = re.sub(r'^```json|^```|```$', '', raw, flags=re.MULTILINE).strip()
                results = json.loads(raw)
                if not isinstance(results, list):
                    return []
                return results  # ✅ success — exit immediately

            except Exception as e:
                err_str    = str(e)
                last_error = err_str

                if '503' in err_str or 'Service Unavailable' in err_str:
                    jitter = random.uniform(0, 1)
                    print(f"    ⚠ 503 on {model} attempt {attempt+1}/{MAX_RETRIES} — waiting {wait:.1f}s")
                    time.sleep(wait + jitter)
                    wait *= 2   # double wait each retry

                elif '429' in err_str or 'rate_limit' in err_str.lower():
                    print(f"    ⚠ 429 quota hit on {model} — switching model")
                    break        # skip remaining retries, jump to next model

                else:
                    print(f"    ✗ Unexpected error ({model}): {err_str[:120]}")
                    return []    # non-retriable — give up on this batch

        print(f"    {model} exhausted — trying next model...")

    print(f"    ✗ All models failed. Last error: {last_error[:120] if last_error else 'unknown'}")
    return []


# ── Main extraction loop ─────────────────────────────────────────────────────
calendar_events   = []
ics_count         = 0
filtered_out      = 0
gemini_calls      = 0
gemini_candidates = []

print(f"Total emails: {len(all_emails)}")
print("—" * 50)

# ── Pass 1: ICS (free, zero tokens) then keyword filter ─────────────────────
for email in all_emails:
    raw_payload = email.get('raw_payload')

    # Try ICS first if raw_payload was stored in Cell 7
    if raw_payload:
        event = parse_ics_from_payload(raw_payload)
        if event:
            event['source_email'] = {
                'account': email['account'],
                'from':    email['from'],
                'subject': email['subject'],
                'date':    email['date'],
            }
            calendar_events.append(event)
            ics_count += 1
            continue  # found via ICS — no need for Gemini

    # Keyword filter — skip emails that almost certainly have no event
    if not likely_has_event(email):
        filtered_out += 1
        continue

    gemini_candidates.append(email)

total_batches = -(-len(gemini_candidates) // BATCH_SIZE)  # ceiling division
print(f"  ICS events found :  {ics_count}  (free, no API tokens)")
print(f"  Filtered out     :  {filtered_out}  (no event keywords)")
print(f"  Sending to Gemini:  {len(gemini_candidates)} emails → {total_batches} batches of ≤{BATCH_SIZE}")
print("—" * 50)

# ── Pass 2: Batched Gemini with retry ───────────────────────────────────────
for batch_start in range(0, len(gemini_candidates), BATCH_SIZE):
    batch_num = batch_start // BATCH_SIZE + 1
    batch     = gemini_candidates[batch_start : batch_start + BATCH_SIZE]

    print(f"  Batch {batch_num}/{total_batches} ({len(batch)} emails)...")
    results = extract_events_batch(batch)
    gemini_calls += 1

    for item in results:
        idx = item.get('email_index')
        if idx is None or not (0 <= idx < len(batch)):
            continue
        source_email         = batch[idx]
        item['source']       = 'gemini'
        item['source_email'] = {
            'account': source_email['account'],
            'from':    source_email['from'],
            'subject': source_email['subject'],
            'date':    source_email['date'],
        }
        calendar_events.append(item)

    print(f"    → {len(results)} events found  |  running total: {len(calendar_events)}")

    # Longer pause between batches to avoid back-to-back 503s
    pause = 3.0 if batch_num > 1 else 1.0
    time.sleep(pause)

# ── Summary ──────────────────────────────────────────────────────────────────
print("—" * 50)
print(f"✅ Done — {len(calendar_events)} calendar events extracted")
print(f"   ICS (free) : {ics_count}")
print(f"   Gemini     : {len(calendar_events) - ics_count}  (across {gemini_calls} API calls)")

Total emails: 200
——————————————————————————————————————————————————
  ICS events found :  0  (free, no API tokens)
  Filtered out     :  143  (no event keywords)
  Sending to Gemini:  57 emails → 8 batches of ≤8
——————————————————————————————————————————————————
  Batch 1/8 (8 emails)...
    → 2 events found  |  running total: 2
  Batch 2/8 (8 emails)...
    → 4 events found  |  running total: 6
  Batch 3/8 (8 emails)...
    → 1 events found  |  running total: 7
  Batch 4/8 (8 emails)...
    → 4 events found  |  running total: 11
  Batch 5/8 (8 emails)...
    ✗ Unexpected error (llama-3.1-8b-instant): Unterminated string starting at: line 30 column 20 (char 652)
    → 0 events found  |  running total: 11
  Batch 6/8 (8 emails)...
    → 5 events found  |  running total: 16
  Batch 7/8 (8 emails)...
    → 2 events found  |  running total: 18
  Batch 8/8 (1 emails)...
    ✗ Unexpected error (llama-3.1-8b-instant): Extra data: line 3 column 1 (char 5)
    → 0 events found  |  running tota

In [ ]:
# ── Cell 8: Chunk emails (sliding window — handles long emails properly) ────────

def chunk_emails(emails, chunk_size=800, overlap=150):
    """
    Convert emails to text chunks with sliding window.
    Each email can produce multiple chunks if the body is long.
    """
    all_chunks = []

    for email in emails:
        # Build the full text for this email
        header = (
            f"Gmail Account: {email['account']}\n"
            f"From: {email['from']}\n"
            f"To: {email['to']}\n"
            f"Date: {email['date']}\n"
            f"Subject: {email['subject']}\n\n"
        )
        body = email.get('body', '')
        full_text = header + body

        if len(full_text.strip()) < 30:
            continue  # skip near-empty emails

        # Sliding window: create overlapping chunks
        start = 0
        while start < len(full_text):
            end = start + chunk_size
            chunk = full_text[start:end].strip()
            if chunk:
                all_chunks.append(chunk)
            if end >= len(full_text):
                break
            start += chunk_size - overlap  # step forward with overlap

    print(f"Total chunks created: {len(all_chunks)} ✅")
    print(f"(from {len(emails)} emails across {len(set(e['account'] for e in emails))} account(s))")
    return all_chunks


chunks = chunk_emails(all_emails)
print("\n--- Sample Chunk ---")
print(chunks[0][:400] if chunks else "No chunks found")

Total chunks created: 704 ✅
(from 200 emails across 2 account(s))

--- Sample Chunk ---
Gmail Account: apekshakor05@gmail.com
From: Zomato <noreply@mailers.zomato.com>
To: apekshakor05@gmail.com
Date: Mon, 20 Apr 2026 10:17:49 +0000 (UTC)
Subject: One day...

<!DOCTYPE html><html xmlns:v="urn:schemas-microsoft-com:vml" xmlns:o="urn:schemas-microsoft-com:office:office" lang="en"><head><title></title><meta http-equiv="Content-Type" content="text/html; charset=utf-8"><meta name="viewpor


In [ ]:
# ── Cell 9: Build vector store using LOCAL embeddings (no API rate limits) ─────
import numpy as np

class SimpleVectorStore:
    def __init__(self):
        self.vectors = None
        self.texts = []

    def add_documents(self, texts, batch_size=64):
        """
        Embed all chunks locally using sentence-transformers.
        batch_size=64 is safe for Colab free tier RAM.
        """
        if not texts:
            print("No chunks to embed.")
            return

        print(f"Embedding {len(texts)} chunks locally ")
        self.texts = texts

        # encode() handles batching internally and shows a progress bar
        self.vectors = embedding_model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True
        )
        print(f"\nVector store ready: {self.vectors.shape[0]} chunks, "
              f"{self.vectors.shape[1]}-dim embeddings ✅")

    def search(self, query, k=5):
        """Cosine similarity search."""
        if self.vectors is None or len(self.texts) == 0:
            return []

        query_vec = embedding_model.encode([query], convert_to_numpy=True)[0]

        # Normalise for cosine similarity
        norm_vecs = self.vectors / (np.linalg.norm(self.vectors, axis=1, keepdims=True) + 1e-10)
        norm_q   = query_vec  / (np.linalg.norm(query_vec) + 1e-10)

        scores   = np.dot(norm_vecs, norm_q)
        top_k    = np.argsort(scores)[-k:][::-1]

        return [
            {"text": self.texts[i], "score": float(scores[i])}
            for i in top_k
        ]


vector_store = SimpleVectorStore()
vector_store.add_documents(chunks)

Embedding 704 chunks locally (no API limits)...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]


Vector store ready: 704 chunks, 384-dim embeddings ✅


In [ ]:
# ── Cell 9b: Cross-encoder re-ranker (local, free, no API) ──────────────────
# Insert this cell right after Cell 9 (vector store).
# It upgrades search quality: bi-encoder retrieves candidates fast,
# cross-encoder re-ranks them precisely. Zero API cost.

try:
    from sentence_transformers import CrossEncoder
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'sentence-transformers'], check=True)
    from sentence_transformers import CrossEncoder

print("Loading cross-encoder re-ranker (~85MB one-time download)...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Re-ranker ready ✅")


def search_and_rerank(query: str, k_retrieve: int = 20, k_final: int = 5):
    """
    Two-stage retrieval:
      1. Bi-encoder (vector_store) fetches k_retrieve candidates fast.
      2. Cross-encoder re-ranks them and returns top k_final.

    k_retrieve should be 3–4× k_final so the re-ranker has good candidates to work with.
    """
    # Stage 1 — fast approximate retrieval
    candidates = vector_store.search(query, k=k_retrieve)
    if not candidates:
        return []

    # Stage 2 — precise re-ranking
    pairs  = [(query, doc['text']) for doc in candidates]
    scores = reranker.predict(pairs)          # numpy array of floats

    # Attach new scores and sort descending
    for doc, score in zip(candidates, scores):
        doc['rerank_score'] = float(score)

    reranked = sorted(candidates, key=lambda d: d['rerank_score'], reverse=True)
    return reranked[:k_final]


# ── Update generate_rag_answer to show rerank scores ────────────────────────
def generate_rag_answer(query, retrieved_docs):
    """Identical interface — just uses rerank_score if available."""
    if not retrieved_docs:
        return "No relevant emails found to answer your question."

    context_text = "\n\n".join([
        f"[Context {i+1} | score: {doc.get('rerank_score', doc['score']):.3f}]\n{doc['text']}"
        for i, doc in enumerate(retrieved_docs)
    ])

    prompt = f"""You are a helpful AI assistant with access to a user's emails from multiple Gmail accounts.
Use ONLY the email context below to answer the question accurately.
If the answer is not clearly in the context, say so honestly.
Always mention which account or sender the information came from when relevant.

User Question: {query}

--- Email Context ---
{context_text}
--- End of Context ---

Answer:"""

    response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=1000
    )
    return response.choices[0].message.content



# ── Quick smoke test ─────────────────────────────────────────────────────────
test_query = "who sent me emails?"
results    = search_and_rerank(test_query, k_retrieve=20, k_final=5)
print(f"\nRe-ranked top 5 results for: '{test_query}'")
for i, r in enumerate(results):
    print(f"  {i+1}. rerank={r['rerank_score']:.3f}  bi-encoder={r['score']:.3f}  |  {r['text'][:80]}")

Loading cross-encoder re-ranker (~85MB one-time download)...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Re-ranker ready ✅

Re-ranked top 5 results for: 'who sent me emails?'
  1. rerank=-5.554  bi-encoder=0.404  |  shakor07@gmail.com. To opt out of future emails, 
clickunsubscribe 
<https://l
  2. rerank=-6.870  bi-encoder=0.380  |  Gmail Account: apekshakor05@gmail.com
From: Akanksha Kor via LinkedIn <messaging
  3. rerank=-7.359  bi-encoder=0.425  |  m_medium=customerio&utm_campaign=abandoned_cart&utm_content=email_A )

Best re
  4. rerank=-7.589  bi-encoder=0.450  |  ge_discover_01-SecurityHelp-0-textfooterglimmer-null-k0yxwk~mo21tt5v~fc-null-nul
  5. rerank=-8.148  bi-encoder=0.417  |  mail_email_member_message_v2%3BWKK0IxN%2FQ%2BOm7xeIFHOz5g%3D%3D&midToken=AQGG0GO


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# ═══════════════════════════════════════════════════════════════════════════════
#  STYLING
# ═══════════════════════════════════════════════════════════════════════════════
STYLE  = {'description_width': 'initial'}
LAYOUT = widgets.Layout

header_html = widgets.HTML("""
<div style="
  background: linear-gradient(135deg, #1a73e8 0%, #0d47a1 100%);
  border-radius: 12px;
  padding: 20px 24px;
  margin-bottom: 8px;
">
  <h2 style="color:white; margin:0; font-size:22px; font-weight:600;">
    📧 Gmail RAG Assistant
  </h2>
  <p style="color:rgba(255,255,255,0.82); margin:6px 0 0; font-size:14px;">
    Ask anything about your emails across all connected accounts
  </p>
</div>
""")

# ═══════════════════════════════════════════════════════════════════════════════
#  TAB 1 — ASK A QUESTION
# ═══════════════════════════════════════════════════════════════════════════════

query_input = widgets.Textarea(
    placeholder="e.g. Who sent me emails last week?  /  Summarise unread emails  /  Any emails about payments?",
    layout=LAYOUT(width='100%', height='80px'),
    style=STYLE,
)

top_k_slider = widgets.IntSlider(
    value=5, min=1, max=15, step=1,
    description='Context chunks (k):',
    style=STYLE,
    layout=LAYOUT(width='420px'),
)

ask_btn  = widgets.Button(description='Ask ✦',  button_style='primary',
                          layout=LAYOUT(width='120px', height='36px'))
clear_btn = widgets.Button(description='Clear',  button_style='',
                           layout=LAYOUT(width='80px',  height='36px'))

answer_out = widgets.Output(layout=LAYOUT(
    border='1px solid #e0e0e0', border_radius='10px',
    min_height='80px', padding='12px', margin_top='8px',
))

retrieved_out = widgets.Output(layout=LAYOUT(
    border='1px solid #e0e0e0', border_radius='10px',
    min_height='40px', padding='12px', margin_top='4px',
))

def on_ask(b):
    q = query_input.value.strip()
    if not q:
        with answer_out:
            clear_output()
            print("⚠️  Please enter a question first.")
        return

    ask_btn.disabled = True
    ask_btn.description = 'Thinking…'

    with answer_out:
        clear_output()
        print("🔍  Searching your emails…")

    with retrieved_out:
        clear_output()

    try:
        docs = search_and_rerank(
            q,
            k_retrieve=top_k_slider.value * 4,
            k_final=top_k_slider.value
        )

        answer = generate_rag_answer(q, docs)

        with answer_out:
            clear_output()
            display(widgets.HTML(f"""
            <div style="font-family:sans-serif;">
              <div style="font-size:12px; color:#888; margin-bottom:6px;">Answer</div>
              <div style="font-size:15px; line-height:1.7; color:#212121; white-space:pre-wrap;">{answer}</div>
            </div>
            """))

        with retrieved_out:
            clear_output()
            chunks_html = ""

            for i, doc in enumerate(docs):
                score_pct = int(doc['score'] * 100)
                bar_color = "#1a73e8" if score_pct >= 60 else "#fbbc04" if score_pct >= 40 else "#ea4335"

                first_line = doc['text'].split('\n')[0][:80]
                preview = doc['text'][:300].replace('<', '&lt;').replace('>', '&gt;')

                chunks_html += f"""
                <details style="margin-bottom:8px; border:1px solid #e8eaf6;
                                border-radius:8px; overflow:hidden;">
                  <summary style="padding:10px 14px; cursor:pointer; background:#f8f9fa;
                                  font-size:13px; font-weight:500; color:#3c4043;
                                  list-style:none; display:flex; justify-content:space-between; align-items:center;">
                    <span>Chunk {i+1} &nbsp;·&nbsp; {first_line}</span>
                    <span style="display:flex; align-items:center; gap:8px;">
                      <span style="background:{bar_color}20; color:{bar_color};
                                   border-radius:20px; padding:2px 10px; font-size:12px;
                                   font-weight:600;">{score_pct}% match</span>
                    </span>
                  </summary>
                  <div style="padding:10px 14px; font-size:12px; font-family:monospace;
                              color:#555; white-space:pre-wrap; background:white;">{preview}{'…' if len(doc['text'])>300 else ''}</div>
                </details>
                """

            display(widgets.HTML(f"""
            <div style="font-family:sans-serif;">
              <div style="font-size:12px; color:#888; margin-bottom:8px;">
                Retrieved {len(docs)} context chunks
              </div>
              {chunks_html}
            </div>
            """))

    except Exception as e:
        with answer_out:
            clear_output()
            print(f"❌ Error: {e}")

    finally:
        ask_btn.disabled = False
        ask_btn.description = 'Ask ✦'

def on_clear(b):
    query_input.value = ''
    with answer_out:   clear_output()
    with retrieved_out: clear_output()

ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)

tab1 = widgets.VBox([
    widgets.HTML("<p style='font-size:13px; color:#555; margin:8px 0 4px;'>"
                 "Type your question below and press <b>Ask</b>.</p>"),
    query_input,
    widgets.HBox([top_k_slider, ask_btn, clear_btn],
                 layout=LAYOUT(align_items='center', gap='12px', margin_top='6px')),
    widgets.HTML("<p style='font-size:12px; color:#888; margin:10px 0 2px;'>Answer</p>"),
    answer_out,
    widgets.HTML("<p style='font-size:12px; color:#888; margin:10px 0 2px;'>"
                 "Retrieved chunks (click to expand)</p>"),
    retrieved_out,
], layout=LAYOUT(padding='4px'))

# ═══════════════════════════════════════════════════════════════════════════════
#  TAB 2 — FETCH & REBUILD
# ═══════════════════════════════════════════════════════════════════════════════

max_emails_input = widgets.BoundedIntText(
    value=100, min=1, max=500,
    description='Max emails / account:',
    style=STYLE, layout=LAYOUT(width='280px'),
)

query_filter_input = widgets.Text(
    value='',
    placeholder="Gmail query filter, e.g. is:unread  newer_than:30d",
    description='Gmail query:',
    style=STYLE, layout=LAYOUT(width='100%'),
)

scope_dropdown = widgets.Dropdown(
    options=[
        ('All mail',       ''),
        ('Inbox only',     'in:inbox'),
        ('Unread only',    'is:unread'),
        ('Starred',        'is:starred'),
        ('Last 7 days',    'newer_than:7d'),
        ('Last 30 days',   'newer_than:30d'),
        ('Last 90 days',   'newer_than:90d'),
    ],
    description='Scope:',
    style=STYLE, layout=LAYOUT(width='260px'),
)

fetch_btn   = widgets.Button(description='Fetch & Rebuild index ↺',
                             button_style='warning',
                             layout=LAYOUT(width='220px', height='36px'))
fetch_out   = widgets.Output(layout=LAYOUT(
    border='1px solid #e0e0e0', border_radius='10px',
    min_height='60px', padding='12px', margin_top='8px',
))

# Per-account progress bars (built dynamically)
progress_box = widgets.VBox([])

def build_progress_bars(account_list):
    rows = []
    for acc in account_list:
        label = widgets.HTML(
            f"<span style='font-size:13px; min-width:220px; display:inline-block;"
            f"color:#3c4043;'>{acc}</span>"
        )
        bar  = widgets.IntProgress(value=0, min=0, max=100,
                                   layout=LAYOUT(width='240px', height='14px'))
        cnt  = widgets.HTML("<span style='font-size:12px; color:#888;'>0 emails</span>")
        rows.append(widgets.HBox([label, bar, cnt],
                                 layout=LAYOUT(align_items='center', gap='10px')))
    return rows, [r.children[1] for r in rows], [r.children[2] for r in rows]

def on_fetch(b):
    fetch_btn.disabled = True
    fetch_btn.description = 'Fetching…'

    with fetch_out:
        clear_output()

    # Build per-account bars
    account_list = list(services.keys())
    rows, bars, counters = build_progress_bars(account_list)
    progress_box.children = [
        widgets.HTML("<p style='font-size:12px; color:#888; margin:8px 0 4px;'>Per-account progress</p>"),
        *rows
    ]

    final_query = ' '.join(filter(None, [
        scope_dropdown.value,
        query_filter_input.value.strip()
    ]))

    new_emails = []
    try:
        for idx, (account, service) in enumerate(services.items()):
            with fetch_out:
                print(f"  Fetching: {account}")

            fetched = fetch_emails_from_account(
                service, account,
                query=final_query,
                max_results=max_emails_input.value
            )
            new_emails.extend(fetched)

            bars[idx].value    = 100
            counters[idx].value = (
                f"<span style='font-size:12px; color:#1a73e8; font-weight:500;'>"
                f"{len(fetched)} emails ✅</span>"
            )
            with fetch_out:
                print(f"  [{account}] {len(fetched)} emails fetched ✅")

        # Rebuild chunks & vector store
        with fetch_out:
            print(f"\nTotal emails: {len(new_emails)}")
            print("Chunking…")

        new_chunks = chunk_emails(new_emails)

        with fetch_out:
            print(f"Total chunks: {len(new_chunks)}")
            print("Re-embedding (local model)…")

        vector_store.add_documents(new_chunks)

        with fetch_out:
            print(f"\n✅ Index rebuilt — {len(new_chunks)} chunks ready for Q&A!")

        # Refresh stats tab
        refresh_stats(new_emails, new_chunks)

    except Exception as e:
        with fetch_out:
            print(f"\n❌ Error during fetch: {e}")
    finally:
        fetch_btn.disabled    = False
        fetch_btn.description = 'Fetch & Rebuild index ↺'

fetch_btn.on_click(on_fetch)

tab2 = widgets.VBox([
    widgets.HTML("<p style='font-size:13px; color:#555; margin:8px 0 4px;'>"
                 "Adjust fetch settings and rebuild the vector index.</p>"),
    widgets.HBox([scope_dropdown, max_emails_input],
                 layout=LAYOUT(gap='16px', align_items='center')),
    query_filter_input,
    fetch_btn,
    progress_box,
    fetch_out,
], layout=LAYOUT(padding='4px'))

# ═══════════════════════════════════════════════════════════════════════════════
#  TAB 3 — STATS
# ═══════════════════════════════════════════════════════════════════════════════

stats_out = widgets.Output(layout=LAYOUT(padding='4px'))

def refresh_stats(email_list=None, chunk_list=None):
    el = email_list if email_list is not None else all_emails
    cl = chunk_list  if chunk_list  is not None else chunks

    from collections import Counter
    per_account = Counter(e['account'] for e in el)
    total_emails = len(el)
    total_chunks = len(cl)
    num_accounts = len(per_account)

    rows_html = ""
    for acc, cnt in per_account.items():
        pct  = round(cnt / total_emails * 100) if total_emails else 0
        rows_html += f"""
        <tr>
          <td style="padding:8px 12px; font-size:13px; color:#3c4043;">{acc}</td>
          <td style="padding:8px 12px; font-size:13px; text-align:right;">{cnt}</td>
          <td style="padding:8px 12px;">
            <div style="background:#e8f0fe; border-radius:4px; height:8px; width:140px;">
              <div style="background:#1a73e8; border-radius:4px; height:8px; width:{pct*1.4:.0f}px;"></div>
            </div>
          </td>
          <td style="padding:8px 12px; font-size:12px; color:#888;">{pct}%</td>
        </tr>
        """

    avg_chunks_per_email = round(total_chunks / total_emails, 1) if total_emails else 0

    with stats_out:
        clear_output()
        display(widgets.HTML(f"""
        <div style="font-family:sans-serif;">
          <div style="display:grid; grid-template-columns:repeat(4,1fr); gap:12px; margin-bottom:16px;">
            <div style="background:#e8f0fe; border-radius:10px; padding:14px 16px;">
              <div style="font-size:24px; font-weight:600; color:#1a73e8;">{num_accounts}</div>
              <div style="font-size:12px; color:#5f6368; margin-top:2px;">accounts</div>
            </div>
            <div style="background:#e6f4ea; border-radius:10px; padding:14px 16px;">
              <div style="font-size:24px; font-weight:600; color:#1e8e3e;">{total_emails}</div>
              <div style="font-size:12px; color:#5f6368; margin-top:2px;">emails fetched</div>
            </div>
            <div style="background:#fce8e6; border-radius:10px; padding:14px 16px;">
              <div style="font-size:24px; font-weight:600; color:#d93025;">{total_chunks}</div>
              <div style="font-size:12px; color:#5f6368; margin-top:2px;">chunks indexed</div>
            </div>
            <div style="background:#fef7e0; border-radius:10px; padding:14px 16px;">
              <div style="font-size:24px; font-weight:600; color:#f29900;">{avg_chunks_per_email}</div>
              <div style="font-size:12px; color:#5f6368; margin-top:2px;">avg chunks / email</div>
            </div>
          </div>

          <div style="font-size:13px; font-weight:500; color:#3c4043; margin-bottom:8px;">Per-account breakdown</div>
          <table style="width:100%; border-collapse:collapse; border:1px solid #e0e0e0; border-radius:8px; overflow:hidden;">
            <thead>
              <tr style="background:#f8f9fa;">
                <th style="padding:8px 12px; text-align:left; font-size:12px; color:#5f6368; font-weight:500;">Account</th>
                <th style="padding:8px 12px; text-align:right; font-size:12px; color:#5f6368; font-weight:500;">Emails</th>
                <th style="padding:8px 12px; font-size:12px; color:#5f6368; font-weight:500;">Distribution</th>
                <th style="padding:8px 12px; font-size:12px; color:#5f6368; font-weight:500;">Share</th>
              </tr>
            </thead>
            <tbody>{rows_html}</tbody>
          </table>
        </div>
        """))

refresh_stats()  # show current state on load

tab3 = widgets.VBox([
    widgets.HTML("<p style='font-size:13px; color:#555; margin:8px 0 4px;'>"
                 "Overview of what's currently indexed. Re-fetching updates this automatically.</p>"),
    stats_out,
], layout=LAYOUT(padding='4px'))

# ═══════════════════════════════════════════════════════════════════════════════
#  TAB 4 — CALENDAR
# ═══════════════════════════════════════════════════════════════════════════════
from collections import defaultdict

cal_out = widgets.Output(layout=LAYOUT(padding='4px'))

def render_calendar():
    # Group events by date
    by_date = defaultdict(list)
    for ev in calendar_events:
        d = ev.get("event_date")
        if d:
            by_date[d].append(ev)

    if not by_date:
        with cal_out:
            clear_output()
            print("No events found. Run Cell 7b first.")
        return

    sorted_dates = sorted(by_date.keys())
    event_rows = ""
    for date in sorted_dates:
        for ev in by_date[date]:
            conflict = sum(
                1 for other in by_date[date]
                if other.get("event_time") == ev.get("event_time") and other != ev
            ) > 0
            badge = "<span style='background:#fce8e6;color:#d93025;border-radius:4px;padding:1px 6px;font-size:11px;margin-left:6px;'>⚠ conflict</span>" if conflict else ""
            event_rows += f"""
            <tr style="border-bottom:1px solid #f1f3f4;">
              <td style="padding:10px 12px;font-size:13px;color:#3c4043;white-space:nowrap;">{date}</td>
              <td style="padding:10px 12px;font-size:13px;color:#888;">{ev.get('event_time') or '—'}</td>
              <td style="padding:10px 12px;font-size:13px;font-weight:500;color:#1a73e8;">{ev.get('title') or 'Untitled'}{badge}</td>
              <td style="padding:10px 12px;font-size:12px;color:#5f6368;">{ev.get('location') or '—'}</td>
              <td style="padding:10px 12px;font-size:12px;color:#5f6368;">{ev['source_email']['from']}</td>
            </tr>"""

    with cal_out:
        clear_output()
        display(widgets.HTML(f"""
        <div style="font-family:sans-serif;">
          <div style="font-size:13px;color:#555;margin-bottom:12px;">
            {len(calendar_events)} events found across {len(by_date)} dates
          </div>
          <table style="width:100%;border-collapse:collapse;border:1px solid #e0e0e0;border-radius:8px;overflow:hidden;">
            <thead>
              <tr style="background:#f8f9fa;">
                <th style="padding:8px 12px;text-align:left;font-size:12px;color:#5f6368;font-weight:500;">Date</th>
                <th style="padding:8px 12px;text-align:left;font-size:12px;color:#5f6368;font-weight:500;">Time</th>
                <th style="padding:8px 12px;text-align:left;font-size:12px;color:#5f6368;font-weight:500;">Event</th>
                <th style="padding:8px 12px;text-align:left;font-size:12px;color:#5f6368;font-weight:500;">Location</th>
                <th style="padding:8px 12px;text-align:left;font-size:12px;color:#5f6368;font-weight:500;">From</th>
              </tr>
            </thead>
            <tbody>{event_rows}</tbody>
          </table>
        </div>
        """))

refresh_cal_btn = widgets.Button(description='Refresh calendar ↺',
                                  button_style='info',
                                  layout=LAYOUT(width='180px', height='36px'))
refresh_cal_btn.on_click(lambda b: render_calendar())
render_calendar()

tab4 = widgets.VBox([
    widgets.HTML("<p style='font-size:13px;color:#555;margin:8px 0 4px;'>Events extracted from your emails. Re-run Cell 7b after fetching new emails.</p>"),
    refresh_cal_btn,
    cal_out,
], layout=LAYOUT(padding='4px'))

# ═══════════════════════════════════════════════════════════════════════════════
#  ASSEMBLE TABS
# ═══════════════════════════════════════════════════════════════════════════════

tabs = widgets.Tab(children=[tab1, tab2, tab3,tab4])
tabs.set_title(0, '💬  Ask')
tabs.set_title(1, '🔄  Fetch & Rebuild')
tabs.set_title(2, '📊  Stats')
tabs.set_title(3, '📅  Calendar')
display(widgets.VBox([
    header_html,
    tabs,
], layout=LAYOUT(width='100%', padding='4px')))

In [ ]:
# ── Cell 13: Re-authenticate with Gmail SEND permission ───────────────────────
# We need an additional OAuth scope to send emails.
# This cell adds 'gmail.send' scope and refreshes tokens for all accounts.
# Run this ONCE — tokens are saved, so you won't need to re-auth every session.

import os, pickle
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.auth.transport.requests import Request

# ⚠️  Both scopes required — readonly for fetching, send for sending
SCOPES_WITH_SEND = [
    'https://www.googleapis.com/auth/gmail.readonly',
    'https://www.googleapis.com/auth/gmail.send',
]

def get_gmail_service_with_send(account_email):
    """Like get_gmail_service() but includes the send scope."""
    safe_name  = account_email.replace('@', '_at_').replace('.', '_')
    token_file = f"token_send_{safe_name}.pickle"   # separate file so readonly tokens stay intact

    creds = None
    if os.path.exists(token_file):
        with open(token_file, 'rb') as f:
            creds = pickle.load(f)

    if creds and creds.expired and creds.refresh_token:
        try:
            creds.refresh(Request())
            with open(token_file, 'wb') as f:
                pickle.dump(creds, f)
            print(f"  Token refreshed for {account_email} ✅")
            return build('gmail', 'v1', credentials=creds)
        except Exception:
            creds = None

    if not creds or not creds.valid:
        flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES_WITH_SEND)
        flow.redirect_uri = 'urn:ietf:wg:oauth:2.0:oob'

        auth_url, _ = flow.authorization_url(
            prompt='consent',
            access_type='offline',
            login_hint=account_email
        )

        print(f"\n{'='*65}")
        print(f"  Authorizing SEND access: {account_email}")
        print(f"  Open this URL (sign in as {account_email}):\n")
        print(f"  {auth_url}\n")
        print('='*65)

        code = input(f"  Paste the authorization code for {account_email}: ").strip()
        flow.fetch_token(code=code)
        creds = flow.credentials

        with open(token_file, 'wb') as f:
            pickle.dump(creds, f)

    return build('gmail', 'v1', credentials=creds)


# Authenticate all accounts with send permission
send_services = {}
for account in GMAIL_ACCOUNTS:
    print(f"\nConnecting (send scope): {account}")
    try:
        send_services[account] = get_gmail_service_with_send(account)
        print(f"✅ Send-enabled: {account}")
    except Exception as e:
        print(f"❌ Failed for {account}: {e}")

print(f"\n{len(send_services)}/{len(GMAIL_ACCOUNTS)} accounts ready to send ✅")



Connecting (send scope): apekshakor05@gmail.com

  Authorizing SEND access: apekshakor05@gmail.com
  Open this URL (sign in as apekshakor05@gmail.com):

  https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=817999543115-r08qv85q4ebqvgo2t9tsmj8ur6mpj39l.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.send&state=BzftOXSPUAddJ1ddgi1duI7waL0Xva&code_challenge=SJ2BaEPZclUSOASD4XST_A2yUnPZTlneBjila8hoKq0&code_challenge_method=S256&prompt=consent&access_type=offline&login_hint=apekshakor05%40gmail.com

  Paste the authorization code for apekshakor05@gmail.com: 4/1Aci98E-QBFFUJGW7lI_lipfKKyG6QbbvecKZLa2bJOFoAAhOyJNDVCf1Nbg
✅ Send-enabled: apekshakor05@gmail.com

Connecting (send scope): apekshakor07@gmail.com

  Authorizing SEND access: apekshakor07@gmail.com
  Open this URL (sign in as apekshakor07@gmail.com):

  https://accounts.google

In [ ]:
# ── Cell 14: Email send helpers + AI reply generator ──────────────────────────

import base64
import re
from email.mime.text      import MIMEText
from email.mime.multipart import MIMEMultipart

# ── 1. Build a raw RFC-2822 message ───────────────────────────────────────────

def build_mime_message(sender, to, subject, body_text, reply_to_msg_id=None, thread_id=None):
    """
    Returns a dict ready for the Gmail API messages.send() call.
    Pass reply_to_msg_id and thread_id to keep the reply in the same thread.
    """
    msg = MIMEMultipart('alternative')
    msg['From']    = sender
    msg['To']      = to
    msg['Subject'] = subject

    if reply_to_msg_id:
        msg['In-Reply-To'] = reply_to_msg_id
        msg['References']  = reply_to_msg_id

    # Plain-text part
    msg.attach(MIMEText(body_text, 'plain', 'utf-8'))

    raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()
    payload = {'raw': raw}
    if thread_id:
        payload['threadId'] = thread_id
    return payload


# ── 2. Send via Gmail API ──────────────────────────────────────────────────────

def send_email(from_account, to, subject, body_text,
               reply_to_msg_id=None, thread_id=None):
    """
    Send an email from `from_account` (must be in send_services).
    Returns the sent message object or raises on failure.
    """
    if from_account not in send_services:
        raise ValueError(f"No send-enabled service for {from_account}. "
                         f"Run Cell 13 first.")

    payload = build_mime_message(
        sender         = from_account,
        to             = to,
        subject        = subject,
        body_text      = body_text,
        reply_to_msg_id= reply_to_msg_id,
        thread_id      = thread_id,
    )

    service = send_services[from_account]
    sent    = service.users().messages().send(userId='me', body=payload).execute()
    print(f"✅ Email sent!  Message ID: {sent['id']}")
    return sent


# ── 3. AI reply / compose generator ───────────────────────────────────────────

def generate_reply(
    original_email: dict,
    user_instruction: str,
    tone: str = "professional",
    from_account: str = "",
) -> str:
    """
    Use Gemini to draft a reply to `original_email` given `user_instruction`.

    original_email keys expected: from, subject, date, body
    tone options: professional | friendly | formal | concise
    """
    prompt = f"""You are an email assistant helping the user ({from_account}) write a reply.

ORIGINAL EMAIL:
From:    {original_email.get('from', '')}
Date:    {original_email.get('date', '')}
Subject: {original_email.get('subject', '')}

{original_email.get('body', '')[:1500]}

---
USER INSTRUCTION:
{user_instruction}

TONE: {tone}

Write ONLY the reply email body (no subject line, no "Here is a reply:" preamble).
Start directly with the salutation (e.g. "Hi [Name],").
End with a professional sign-off followed by the sender's first name only.
Keep it concise and natural."""

    response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=500
    )
    return response.choices[0].message.content.strip()


def generate_compose(
    user_instruction: str,
    tone: str = "professional",
    from_account: str = "",
) -> dict:
    """
    Use Gemini to compose a brand-new email from `user_instruction`.
    Returns {'subject': ..., 'body': ...}
    """
    prompt = f"""You are an email assistant helping {from_account} compose a new email.

USER INSTRUCTION:
{user_instruction}

TONE: {tone}

Respond in this EXACT format (no extra text):
SUBJECT: <subject line here>
BODY:
<email body here starting with salutation>"""

    response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=600
)
    text = response.choices[0].message.content.strip()


    # Parse subject and body
    subject, body = "", text
    match = re.search(r'SUBJECT:\s*(.+?)\nBODY:\n(.*)', text, re.DOTALL)
    if match:
        subject = match.group(1).strip()
        body    = match.group(2).strip()

    return {'subject': subject, 'body': body}


# ── 4. Extract reply-to address from a raw "From" header ──────────────────────

def extract_email_address(from_header: str) -> str:
    """'John Doe <john@example.com>' → 'john@example.com'"""
    match = re.search(r'<(.+?)>', from_header)
    return match.group(1) if match else from_header.strip()


print("Email helpers loaded ✅")
print(f"Send-enabled accounts: {list(send_services.keys())}")


Email helpers loaded ✅
Send-enabled accounts: ['apekshakor05@gmail.com', 'apekshakor07@gmail.com']


In [ ]:
# ── Cell 15: Email Compose & Send UI ──────────────────────────────────────────
# Depends on: all_emails, send_services, generate_reply, generate_compose,
#             send_email, extract_email_address  (from Cells 13 & 14)

import ipywidgets as widgets
from IPython.display import display, clear_output

STYLE  = {'description_width': 'initial'}
LAYOUT = widgets.Layout

# ─── shared state ─────────────────────────────────────────────────────────────
state = {
    'mode':             'reply',   # 'reply' | 'compose'
    'selected_email':   None,      # dict from all_emails
    'generated_subject':'',
    'generated_body':   '',
}

# ═══════════════════════════════════════════════════════════════════════════════
#  HEADER
# ═══════════════════════════════════════════════════════════════════════════════

header_html = widgets.HTML("""
<div style="
  background: linear-gradient(135deg, #1a73e8 0%, #0d47a1 100%);
  border-radius: 12px;
  padding: 20px 24px;
  margin-bottom: 8px;
">
  <h2 style="color:white; margin:0; font-size:22px; font-weight:600;">
    ✉️  AI Email Composer
  </h2>
  <p style="color:rgba(255,255,255,0.82); margin:6px 0 0; font-size:14px;">
    Reply to existing emails or compose new ones — AI drafts, you review & send
  </p>
</div>
""")

# ═══════════════════════════════════════════════════════════════════════════════
#  TAB A — REPLY TO AN EXISTING EMAIL
# ═══════════════════════════════════════════════════════════════════════════════

# ── email picker ──────────────────────────────────────────────────────────────
def make_email_options():
    opts = []
    for i, em in enumerate(all_emails):
        label = f"[{em['account'].split('@')[0]}]  {em['subject'][:55]}  —  {em['from'][:35]}"
        opts.append((label, i))
    return opts if opts else [("No emails loaded", -1)]

email_picker = widgets.Select(
    options     = make_email_options(),
    description = 'Select email:',
    rows        = 8,
    style       = STYLE,
    layout      = LAYOUT(width='100%'),
)

email_preview_out = widgets.Output(layout=LAYOUT(
    border='1px solid #e0e0e0', border_radius='8px',
    padding='10px', min_height='80px', margin_bottom='8px',
))

def show_email_preview(change=None):
    idx = email_picker.value
    with email_preview_out:
        clear_output()
        if idx is None or idx < 0 or idx >= len(all_emails):
            return
        em = all_emails[idx]
        state['selected_email'] = em
        reply_from_dropdown.value = em['account'] if em['account'] in list(send_services.keys()) else list(send_services.keys())[0]
        reply_to_field.value      = extract_email_address(em.get('from', ''))
        reply_subject_field.value = ('Re: ' + em['subject']) if not em['subject'].startswith('Re:') else em['subject']

        display(widgets.HTML(f"""
        <div style="font-family:sans-serif; font-size:13px; line-height:1.6;">
          <table style="border-collapse:collapse; width:100%;">
            <tr><td style="color:#888; width:70px; padding:2px 8px 2px 0;">From</td>
                <td style="color:#212121;">{em.get('from','')}</td></tr>
            <tr><td style="color:#888; padding:2px 8px 2px 0;">Date</td>
                <td style="color:#212121;">{em.get('date','')}</td></tr>
            <tr><td style="color:#888; padding:2px 8px 2px 0;">Subject</td>
                <td style="color:#212121; font-weight:500;">{em.get('subject','')}</td></tr>
            <tr><td style="color:#888; padding:2px 8px 2px 0;">Account</td>
                <td style="color:#1a73e8;">{em.get('account','')}</td></tr>
          </table>
          <div style="margin-top:10px; padding-top:8px; border-top:1px solid #e0e0e0;
                      color:#444; white-space:pre-wrap; max-height:120px; overflow-y:auto;
                      font-size:12px;">{em.get('body','(no body)')[:600]}</div>
        </div>
        """))

email_picker.observe(show_email_preview, names='value')

# ── reply form ────────────────────────────────────────────────────────────────
reply_from_dropdown = widgets.Dropdown(
    options     = list(send_services.keys()) or ['(run Cell 13 first)'],
    description = 'Send from:',
    style       = STYLE,
    layout      = LAYOUT(width='100%'),
)

reply_to_field = widgets.Text(
    description = 'To:',
    style       = STYLE,
    layout      = LAYOUT(width='100%'),
)

reply_subject_field = widgets.Text(
    description = 'Subject:',
    style       = STYLE,
    layout      = LAYOUT(width='100%'),
)

reply_instruction = widgets.Textarea(
    description = 'Your instruction:',
    placeholder = 'e.g. "Politely decline the meeting"  /  "Confirm I will attend"  /  "Ask for more details about the deadline"',
    style       = STYLE,
    layout      = LAYOUT(width='100%', height='70px'),
)

reply_tone = widgets.ToggleButtons(
    options     = ['professional', 'friendly', 'formal', 'concise'],
    description = 'Tone:',
    value       = 'professional',
    style       = {'description_width': 'initial', 'button_width': '110px'},
)

generate_reply_btn = widgets.Button(
    description  = 'Generate draft ✦',
    button_style = 'info',
    layout       = LAYOUT(width='180px', height='36px'),
)

reply_body_field = widgets.Textarea(
    description = 'Draft body:',
    placeholder = 'Your AI-generated draft will appear here. You can edit before sending.',
    style       = STYLE,
    layout      = LAYOUT(width='100%', height='180px'),
)

send_reply_btn = widgets.Button(
    description  = 'Send reply  ✉',
    button_style = 'success',
    layout       = LAYOUT(width='160px', height='36px'),
    disabled     = True,
)

reply_status_out = widgets.Output(layout=LAYOUT(
    border='1px solid #e0e0e0', border_radius='8px',
    padding='8px', min_height='36px',
))

def on_generate_reply(b):
    em = state.get('selected_email')
    if not em:
        with reply_status_out:
            clear_output(); print("⚠️  Select an email first.")
        return
    if not reply_instruction.value.strip():
        with reply_status_out:
            clear_output(); print("⚠️  Enter an instruction for the AI.")
        return

    generate_reply_btn.disabled     = True
    generate_reply_btn.description  = 'Generating…'

    with reply_status_out:
        clear_output(); print("🤖 Generating draft…")

    try:
        draft = generate_reply(
            original_email   = em,
            user_instruction = reply_instruction.value.strip(),
            tone             = reply_tone.value,
            from_account     = reply_from_dropdown.value,
        )
        reply_body_field.value      = draft
        send_reply_btn.disabled     = False
        state['generated_body']     = draft
        with reply_status_out:
            clear_output()
            print("✅ Draft ready — review and edit above, then hit Send.")
    except Exception as e:
        with reply_status_out:
            clear_output(); print(f"❌ Error: {e}")
    finally:
        generate_reply_btn.disabled    = False
        generate_reply_btn.description = 'Generate draft ✦'

def on_send_reply(b):
    if not reply_body_field.value.strip():
        with reply_status_out:
            clear_output(); print("⚠️  Draft body is empty.")
        return

    send_reply_btn.disabled    = True
    send_reply_btn.description = 'Sending…'

    with reply_status_out:
        clear_output(); print("📤 Sending…")

    try:
        send_email(
            from_account = reply_from_dropdown.value,
            to           = reply_to_field.value.strip(),
            subject      = reply_subject_field.value.strip(),
            body_text    = reply_body_field.value.strip(),
        )
        with reply_status_out:
            clear_output()
            display(widgets.HTML("""
            <div style="background:#e6f4ea; border-radius:8px; padding:10px 14px;
                        color:#1e8e3e; font-size:13px; font-weight:500;">
              ✅ Reply sent successfully!
            </div>"""))
        reply_body_field.value     = ''
        send_reply_btn.disabled    = True
        send_reply_btn.description = 'Send reply  ✉'
    except Exception as e:
        with reply_status_out:
            clear_output(); print(f"❌ Send failed: {e}")
        send_reply_btn.disabled    = False
        send_reply_btn.description = 'Send reply  ✉'

generate_reply_btn.on_click(on_generate_reply)
send_reply_btn.on_click(on_send_reply)

# Trigger initial preview
show_email_preview()

tab_reply = widgets.VBox([
    widgets.HTML("<p style='font-size:13px;color:#555;margin:8px 0 4px;'>"
                 "Pick an email to reply to, describe what you want to say, and let AI draft it.</p>"),
    widgets.HTML("<b style='font-size:13px;'>Step 1 — pick an email</b>"),
    email_picker,
    email_preview_out,
    widgets.HTML("<b style='font-size:13px;'>Step 2 — configure reply</b>"),
    reply_from_dropdown,
    reply_to_field,
    reply_subject_field,
    reply_instruction,
    reply_tone,
    generate_reply_btn,
    widgets.HTML("<b style='font-size:13px;'>Step 3 — review & send</b>"),
    reply_body_field,
    widgets.HBox([send_reply_btn],
                 layout=LAYOUT(margin_top='4px')),
    reply_status_out,
], layout=LAYOUT(padding='4px', gap='6px'))


# ═══════════════════════════════════════════════════════════════════════════════
#  TAB B — COMPOSE A NEW EMAIL
# ═══════════════════════════════════════════════════════════════════════════════

compose_from = widgets.Dropdown(
    options     = list(send_services.keys()) or ['(run Cell 13 first)'],
    description = 'Send from:',
    style       = STYLE,
    layout      = LAYOUT(width='100%'),
)

compose_to = widgets.Text(
    description = 'To:',
    placeholder = 'recipient@example.com',
    style       = STYLE,
    layout      = LAYOUT(width='100%'),
)

compose_instruction = widgets.Textarea(
    description = 'What to write:',
    placeholder = 'e.g. "Email Priya asking if she can review my PR by Friday, keep it short"',
    style       = STYLE,
    layout      = LAYOUT(width='100%', height='80px'),
)

compose_tone = widgets.ToggleButtons(
    options     = ['professional', 'friendly', 'formal', 'concise'],
    description = 'Tone:',
    value       = 'professional',
    style       = {'description_width': 'initial', 'button_width': '110px'},
)

generate_compose_btn = widgets.Button(
    description  = 'Generate draft ✦',
    button_style = 'info',
    layout       = LAYOUT(width='180px', height='36px'),
)

compose_subject_field = widgets.Text(
    description = 'Subject:',
    placeholder = 'AI will suggest one — you can edit it',
    style       = STYLE,
    layout      = LAYOUT(width='100%'),
)

compose_body_field = widgets.Textarea(
    description = 'Body:',
    placeholder = 'AI-generated draft will appear here.',
    style       = STYLE,
    layout      = LAYOUT(width='100%', height='200px'),
)

send_compose_btn = widgets.Button(
    description  = 'Send email  ✉',
    button_style = 'success',
    layout       = LAYOUT(width='160px', height='36px'),
    disabled     = True,
)

compose_status_out = widgets.Output(layout=LAYOUT(
    border='1px solid #e0e0e0', border_radius='8px',
    padding='8px', min_height='36px',
))

def on_generate_compose(b):
    if not compose_instruction.value.strip():
        with compose_status_out:
            clear_output(); print("⚠️  Enter an instruction for the AI.")
        return

    generate_compose_btn.disabled    = True
    generate_compose_btn.description = 'Generating…'

    with compose_status_out:
        clear_output(); print("🤖 Generating email…")

    try:
        result = generate_compose(
            user_instruction = compose_instruction.value.strip(),
            tone             = compose_tone.value,
            from_account     = compose_from.value,
        )
        compose_subject_field.value  = result.get('subject', '')
        compose_body_field.value     = result.get('body', '')
        send_compose_btn.disabled    = False

        with compose_status_out:
            clear_output()
            print("✅ Draft ready — review and edit above, then hit Send.")
    except Exception as e:
        with compose_status_out:
            clear_output(); print(f"❌ Error: {e}")
    finally:
        generate_compose_btn.disabled    = False
        generate_compose_btn.description = 'Generate draft ✦'

def on_send_compose(b):
    if not compose_to.value.strip():
        with compose_status_out:
            clear_output(); print("⚠️  Enter a recipient email address.")
        return
    if not compose_body_field.value.strip():
        with compose_status_out:
            clear_output(); print("⚠️  Draft body is empty.")
        return

    send_compose_btn.disabled    = True
    send_compose_btn.description = 'Sending…'

    with compose_status_out:
        clear_output(); print("📤 Sending…")

    try:
        send_email(
            from_account = compose_from.value,
            to           = compose_to.value.strip(),
            subject      = compose_subject_field.value.strip(),
            body_text    = compose_body_field.value.strip(),
        )
        with compose_status_out:
            clear_output()
            display(widgets.HTML("""
            <div style="background:#e6f4ea; border-radius:8px; padding:10px 14px;
                        color:#1e8e3e; font-size:13px; font-weight:500;">
              ✅ Email sent successfully!
            </div>"""))
        compose_to.value             = ''
        compose_subject_field.value  = ''
        compose_body_field.value     = ''
        compose_instruction.value    = ''
        send_compose_btn.disabled    = True
        send_compose_btn.description = 'Send email  ✉'
    except Exception as e:
        with compose_status_out:
            clear_output(); print(f"❌ Send failed: {e}")
        send_compose_btn.disabled    = False
        send_compose_btn.description = 'Send email  ✉'

generate_compose_btn.on_click(on_generate_compose)
send_compose_btn.on_click(on_send_compose)

tab_compose = widgets.VBox([
    widgets.HTML("<p style='font-size:13px;color:#555;margin:8px 0 4px;'>"
                 "Describe the email you want to send — AI writes the subject and body.</p>"),
    compose_from,
    compose_to,
    compose_instruction,
    compose_tone,
    generate_compose_btn,
    widgets.HTML("<b style='font-size:13px;'>Review & edit before sending</b>"),
    compose_subject_field,
    compose_body_field,
    widgets.HBox([send_compose_btn],
                 layout=LAYOUT(margin_top='4px')),
    compose_status_out,
], layout=LAYOUT(padding='4px', gap='6px'))


# ═══════════════════════════════════════════════════════════════════════════════
#  TAB C — SENT HISTORY (this session)
# ═══════════════════════════════════════════════════════════════════════════════

sent_log = []   # list of dicts appended on every successful send

# Monkey-patch send_email to log sends
_original_send_email = send_email

def send_email_with_log(from_account, to, subject, body_text, **kwargs):
    import datetime
    result = _original_send_email(from_account, to, subject, body_text, **kwargs)
    sent_log.append({
        'from':    from_account,
        'to':      to,
        'subject': subject,
        'preview': body_text[:200],
        'time':    datetime.datetime.now().strftime('%H:%M:%S'),
        'id':      result.get('id', ''),
    })
    refresh_sent_tab()
    return result

send_email = send_email_with_log   # replace in scope

sent_tab_out = widgets.Output(layout=LAYOUT(padding='4px'))

def refresh_sent_tab():
    with sent_tab_out:
        clear_output()
        if not sent_log:
            display(widgets.HTML(
                "<p style='font-size:13px;color:#888;margin:16px 0;'>"
                "No emails sent this session yet.</p>"
            ))
            return
        rows = ""
        for s in reversed(sent_log):
            rows += f"""
            <tr style="border-bottom:1px solid #f0f0f0;">
              <td style="padding:10px 12px; font-size:12px; color:#888;">{s['time']}</td>
              <td style="padding:10px 12px; font-size:13px; color:#1a73e8;">{s['from']}</td>
              <td style="padding:10px 12px; font-size:13px;">{s['to']}</td>
              <td style="padding:10px 12px; font-size:13px; font-weight:500;">{s['subject']}</td>
              <td style="padding:10px 12px; font-size:11px; color:#888; max-width:200px;
                         overflow:hidden; text-overflow:ellipsis; white-space:nowrap;">{s['preview']}</td>
            </tr>"""
        display(widgets.HTML(f"""
        <div style="font-family:sans-serif;">
          <div style="font-size:13px; color:#888; margin-bottom:10px;">
            {len(sent_log)} email(s) sent this session
          </div>
          <table style="width:100%; border-collapse:collapse; font-size:13px;
                        border:1px solid #e0e0e0; border-radius:8px; overflow:hidden;">
            <thead>
              <tr style="background:#f8f9fa;">
                <th style="padding:8px 12px; text-align:left; font-size:12px; color:#5f6368; font-weight:500;">Time</th>
                <th style="padding:8px 12px; text-align:left; font-size:12px; color:#5f6368; font-weight:500;">From</th>
                <th style="padding:8px 12px; text-align:left; font-size:12px; color:#5f6368; font-weight:500;">To</th>
                <th style="padding:8px 12px; text-align:left; font-size:12px; color:#5f6368; font-weight:500;">Subject</th>
                <th style="padding:8px 12px; text-align:left; font-size:12px; color:#5f6368; font-weight:500;">Preview</th>
              </tr>
            </thead>
            <tbody>{rows}</tbody>
          </table>
        </div>"""))

refresh_sent_tab()

tab_sent = widgets.VBox([
    widgets.HTML("<p style='font-size:13px;color:#555;margin:8px 0 4px;'>"
                 "Emails sent during this Colab session.</p>"),
    sent_tab_out,
], layout=LAYOUT(padding='4px'))


# ═══════════════════════════════════════════════════════════════════════════════
#  ASSEMBLE
# ═══════════════════════════════════════════════════════════════════════════════

tabs = widgets.Tab(children=[tab_reply, tab_compose, tab_sent])
tabs.set_title(0, '↩  Reply')
tabs.set_title(1, '✏️  Compose')
tabs.set_title(2, '📤  Sent')

display(widgets.VBox([
    header_html,
    tabs,
], layout=LAYOUT(width='100%', padding='4px')))


HTML(value="<p style='font-size:13px;color:#888;margin:16px 0;'>No emails sent this session yet.</p>")